In [ ]:
!python --version

Python 3.12.13


In [4]:
!nvidia-smi

Thu May 21 09:21:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Clone or update the repository

`git pull` only works inside an existing git repository. On a fresh Colab runtime, `/content` is not a repository, so this cell clones the repo the first time and pulls updates on later runs.

In [3]:
%%bash
set -euo pipefail

cd /content
if [ -d lerobot_trossen/.git ]; then
  cd lerobot_trossen
  git pull --ff-only
else
  git clone https://github.com/121philip/lerobot_trossen.git
  cd lerobot_trossen
fi

git rev-parse --show-toplevel
git status --short

Already up to date.
/content/lerobot_trossen


## Install the training environment

Colab does not need a manually activated virtualenv for this workflow. `uv sync` creates `.venv`, and `uv run ...` runs commands inside that environment.

In [4]:
%%bash
set -euo pipefail

cd /content/lerobot_trossen
python -m pip install -q --upgrade pip uv
uv sync --extra smolvla --frozen

uv run python - <<'PY'
import torch

print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
print('torch cuda', torch.version.cuda)
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
PY

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 77.1 MB/s eta 0:00:00
torch 2.7.1+cu126
cuda available True
torch cuda 12.6
gpu Tesla T4


Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Prepared 121 packages in 1m 13s
Installed 121 packages in 917ms
 + accelerate==1.11.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.2
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.12.1
 + attrs==25.4.0
 + av==15.1.0
 + certifi==2025.10.5
 + charset-normalizer==3.4.4
 + click==8.3.0
 + cloudpickle==3.1.1
 + cmake==4.1.2
 + datasets==4.1.1
 + deepdiff==8.6.1
 + diffusers==0.35.2
 + dill==0.4.0
 + docopt==0.6.2
 + draccus==0.10.0
 + einops==0.8.1
 + evdev==1.9.2
 + farama-notifications==0.0.4
 + filelock==3.20.0
 + frozenlist==1.8.0
 + fsspec==2025.9.0
 + gitdb==4.0.12
 + gitpython==3.1.45
 + gymnasium==1.2.1
 + h11==0.16.0
 + hf-xet==1.4.2
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.7.2
 + idna==3.11
 + imageio==2.37.0
 + imageio-ffmpeg==0.6.0
 + importlib-metadata==8.7.0
 + jinja2==3.1.6
 + jsonlines==4.0.0
 + lerobot==0.5.0
 + lerobot-robot-tross

## Optional: train

Before training with `--policy.push_to_hub=true` and `--wandb.enable=true`, set `HF_TOKEN` and `WANDB_API_KEY` in Colab Secrets or log in interactively.

In [5]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

from huggingface_hub import login
import wandb

login(token=os.environ["HF_TOKEN"])
wandb.login(key=os.environ["WANDB_API_KEY"])


TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
%%bash
set -euo pipefail

cd /content/lerobot_trossen
bash important_code/training/fine_tune_smolvla.sh